# 01 - Bronze layer: Event Hub -> Delta (raw)

Reads patient events from **Azure Event Hubs** (Kafka endpoint) with Spark Structured Streaming and appends them,
**unchanged**, to a Delta table in ADLS Gen2. Nothing is cleaned here - bronze is the raw history.

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ---------- Config (non-secret) ----------
STORAGE_ACCOUNT = "storageaccount5326"
CONTAINER = "hospitals"
EVENTHUBS_NAMESPACE = "depi-eventhub.servicebus.windows.net"
EVENT_HUB_NAME = "hospitalstopic"

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
bronze_path = f"{BASE_PATH}/raw_data"
checkpoint_path = f"{BASE_PATH}/_checkpoints/bronze"   # kept OUTSIDE the table folder

SECRET_SCOPE = "hospital-scope"   # Databricks secret scope (see README -> "Secrets")

def get_secret(name):
    """Read a secret from the Databricks secret scope.
    Fallback: a notebook widget (paste the value at runtime - clear it before committing the notebook)."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=name)
    except Exception:
        dbutils.widgets.text(name, "")
        value = dbutils.widgets.get(name)
        if not value:
            raise ValueError(f"Secret '{name}' not found in scope '{SECRET_SCOPE}' and widget '{name}' is empty.")
        return value

: 

In [ ]:
# ---------- Credentials (from secrets, never hard-coded) ----------
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    get_secret("storage-key"),
)
EVENTHUB_CONNECTION_STRING = get_secret("eventhub-listen-conn")   # a LISTEN-only policy is enough here

kafka_options = {
    "kafka.bootstrap.servers": f"{EVENTHUBS_NAMESPACE}:9093",
    "subscribe": EVENT_HUB_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{EVENTHUB_CONNECTION_STRING}";'
    ),
    "startingOffsets": "latest",
    "failOnDataLoss": "false",
}

In [ ]:
streaming_df = spark.readStream.format("kafka").options(**kafka_options).load()

# Keep the raw JSON + the time the event reached Event Hub (used later as event_ingestion_time)
json_stream_df = streaming_df.selectExpr(
    "CAST(value AS STRING) AS json_value",
    "timestamp AS kafka_timestamp",
)

query = (
    json_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")      # lets an older table (json_value only) accept the new kafka_timestamp column
    .start(bronze_path)
)
print("Streaming started - events are being written to:", bronze_path)

In [ ]:
query.awaitTermination()